# Inference and analysis of cell-cell communication using MultiChat (without ATAC-seq data) for L-R identification 

This tutorial provides a step-by-step guide to performing spatial cell–cell communication (CCC) analysis using **MultiChat_for_ligand-receptor_identification**, a specialized mode of the MultiChat framework designed for datasets without matched scATAC-seq data.

Using the ISSAAC_GT-seq (mouse cerebral cortex) dataset as a case study, we illustrate the workflow, including data preprocessing and the model training.

This tutorial is logically divided into two main sections:
1. **Data Preprocessing**: Loading, filtering, and preparing the required transcriptomic data, spatial information, and cell embeddings.
2. **Ligand-receptor interaction inference**: Inferring CCC mediated by L-R pairs.

Let's begin by importing the necessary Python libraries and configuring our environment.

## Import MultiChat and other packages

In [1]:
import numpy
import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", message="Exception ignored on calling ctypes callback function")
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import random
from pathlib import Path
from scipy.sparse import csr_matrix, save_npz, load_npz
from scipy.stats import pearsonr
from typing import Dict, Tuple
from tqdm import tqdm
from collections import defaultdict
import polars as pl
import json
from scipy import stats
import multiprocessing

import MultiChat as MC
print(MC.__version__)

0.2.0


## Part 1: Perform processing

Before training the models, we need to prepare the required input datasets. To help users quickly run this tutorial, we provide [processed data](https://doi.org/10.6084/m9.figshare.30834524) that can be used directly. In this step, we will:
- Load the gene expression (RNA) matrix. 
- Load the ligand-receptor interaction database.
- Load the cell representation matrix (`Cell_rep.csv`), It can be generated during the [data preprocessing workflow](./data_preprocessing_on_ISSAAC.ipynb), replaced with a customized cell representation matrix, or set to the cell expression matrix for a quick test run.
- Filter out unexpressed genes and subset the L-R database to only include pairs where both ligands and receptors are actively expressed in our spatial matrix.
- Run the `Preprocess_CCC_model` function to format the ligand and receptor expression data for the subsequent modules.

In [ ]:
base_path = '/home/nas2/biod/zhencaiwei/MultiChat-main/Datasets/ISSAAC_GT/' # replace with your own path
db = pd.read_csv(os.path.join(base_path, "inputs/LRpairDB_groundtruth_140d_2.csv"), header=0, sep=",")
expmatrix = pd.read_csv(os.path.join(base_path, "inputs/RNAmatrix.csv"), header=0, index_col=0, sep=",")
cell_rep = pd.read_csv(os.path.join(base_path, "inputs/RNAmatrix.csv"), header=0, index_col=0, sep=",").T # 'Cell_rep.csv' can be replaced with a customized cell representation matrix.
non_zero_counts = (expmatrix > 0).sum(axis=1)
expmatrix_filt1 = expmatrix[non_zero_counts >= 5]
split_ligand_symbols = db['Ligand_Symbol'].str.split('_') 
mask_ligand = split_ligand_symbols.apply(lambda symbols: all(symbol in expmatrix_filt1.index for symbol in symbols)) 
db_filt1 = db[mask_ligand] 
split_receptor_symbols = db_filt1['Receptor_Symbol'].str.split('_') 
mask_receptor = split_receptor_symbols.apply(lambda symbols: all(symbol in expmatrix_filt1.index for symbol in symbols)) 
db_filt1 = db_filt1[mask_receptor] 
lig_exp, rec_exp = MC.pp.Preprocess_CCC_model(base_path, db_filt1, cell_rep, expmatrix_filt1)

## Part 2: Infer significant L–R communication signals without ATAC-seq data

In this section, we identify significant ligand–receptor pairs using MultiChat when scATAC-seq data are unavailable.

### 2.1 Train contrastive learning module

First, we load the cell-type annotations and spatial coordinates to identify spatially adjacent "positive pairs" of cells that are physically capable of communicating. 

Then, we initialize the MultiChat contrastive learning module and execute the training process to compute the biological interaction strength between these adjacent cells.

In [ ]:
'''get postive pairs'''
cell_clus = pd.read_csv(base_path+'inputs/celltype_info.csv', header=0, index_col=0, sep="\t")
cell_clus.rename(columns={'celltype': 'cell_type'}, inplace=True)
cell_loc = pd.read_csv(base_path+'inputs/Coord.csv', header=0, index_col=0)
parser                  =  MC.utilities.parameter_setting()
args, unknown           = parser.parse_known_args()
args.inputPath          = '/home/nas2/biod/zhencaiwei/MultiChat-main/Datasets/ISSAAC_GT/' 
args.outPath            = args.inputPath + 'CCC/'
MC.utilities.get_cell_positive_pairs(cell_clus, cell_loc, args)

In [6]:
parser  =  MC.utilities.parameter_setting() 
args, unknown = parser.parse_known_args()

args.gpu_id = 0
if args.use_cuda and torch.cuda.is_available():
    device = torch.device(f'cuda:{args.gpu_id}')
    torch.cuda.set_device(args.gpu_id)
else:
    device = torch.device('cpu')

## random seed 
numpy.random.seed( args.seed )
random.seed( args.seed )
torch.manual_seed( args.seed )
torch.cuda.manual_seed( args.seed )

start = time.time()
args.inputPath          = '/home/nas2/biod/zhencaiwei/MultiChat-main/Datasets/ISSAAC_GT/' # replace with your own path
args.use_cuda           = args.use_cuda and torch.cuda.is_available()

args.outPath            = args.inputPath + 'CCC/'	
args.spatialLocation    = args.inputPath + 'inputs/' + 'Coord.csv'
args.annoFile           = args.inputPath + 'inputs/' + 'celltype_info.csv'
args.pos_pair           = args.outPath + args.pos_pair

args.Ligands_exp        = args.outPath + args.Ligands_exp
args.Receptors_exp      = args.outPath + args.Receptors_exp

args.patience           = 15
args.lr_cci             = 0.001
args.attn_drop          = 0
args.tau                = 0.05

args.selected_cell_type = None
args.InterCCC_Name      = 'LRI_module_strength.txt'

MC.model_training.Train_CCC_model(args)
	
duration = time.time() - start
print('Finish training, total time is: ' + str(duration) + 's' )

spot location for adjacency
loading cell type annotations
Calculating pairwise distances between spots
spot-ligand data
spot-receptor data
Size of CCC pairs: 140
Start model training
Using GPU: 0
0 cost: tensor(349.9706)
10 cost: tensor(101.3115) tensor(0.1173, device='cuda:0', grad_fn=<DivBackward0>)
20 cost: tensor(54.6324) tensor(0.0853, device='cuda:0', grad_fn=<DivBackward0>)
30 cost: tensor(28.5963) tensor(0.0522, device='cuda:0', grad_fn=<DivBackward0>)
40 cost: tensor(16.5791) tensor(0.0484, device='cuda:0', grad_fn=<DivBackward0>)
50 cost: tensor(11.0612) tensor(0.0343, device='cuda:0', grad_fn=<DivBackward0>)
60 cost: tensor(8.1018) tensor(0.0277, device='cuda:0', grad_fn=<DivBackward0>)
70 cost: tensor(6.3238) tensor(0.0219, device='cuda:0', grad_fn=<DivBackward0>)
80 cost: tensor(5.2032) tensor(0.0175, device='cuda:0', grad_fn=<DivBackward0>)
90 cost: tensor(4.4260) tensor(0.0149, device='cuda:0', grad_fn=<DivBackward0>)
100 cost: tensor(3.8407) tensor(0.0134, device='cuda:

### 2.2 Generate L–R background data

> To statistically determine which L–R interactions are biologically meaningful, we need a background (null) distribution.
>
> Here, we create this background by perturbing the positive cell pairs and shuffling the ligand/receptor expression profiles. We then train the MultiChat contrastive learning module on this permuted data multiple times (using multiprocessing for 10 parallel runs) to construct a robust and reliable background distribution.

In [ ]:
'''Step1:get background postive pairs'''
pos_pair = pd.read_csv(base_path + 'CCC/Spot_positive_pairs.txt', header=None, index_col=None, sep="\t")
pos_pair_perturb = pos_pair.apply(MC.utilities.perturb_pos_pair_row, axis=1)
lig_exp = pd.read_csv(base_path + 'CCC/ligands_expression.txt', header=0, index_col=0, sep="\t")
rec_exp = pd.read_csv(base_path + 'CCC/receptors_expression.txt', header=0, index_col=0, sep="\t")
lig_exp_shuffled = lig_exp.apply(np.random.permutation)
rec_exp_shuffled = rec_exp.apply(np.random.permutation)
lig_exp_shuffled.to_csv(base_path + 'Bg_CCC/ligands_expression_shuffled.txt', sep="\t")
rec_exp_shuffled.to_csv(base_path + 'Bg_CCC/receptors_expression_shuffled.txt', sep="\t")
pos_pair_perturb.to_csv(base_path + 'Bg_CCC/Spot_positive_pairs_shuffled.txt', sep="\t", header=False, index=False)

In [ ]:
'''Step2-1:Run MultiChat contrastive learning module to generate 10 sets of background data in parallel'''
def run_training(run_idx):  
    parser = MC.utilities.parameter_setting()
    args, unknown = parser.parse_known_args()
    args.gpu_id = 0
    
    base_seed = 42  
    args.seed = base_seed + run_idx
    
    if args.use_cuda and torch.cuda.is_available():
        device = torch.device(f'cuda:{args.gpu_id}')
        torch.cuda.set_device(device)  
    else:
        device = torch.device('cpu')
    
    np.random.seed(args.seed)
    random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(args.seed)
    
    start = time.time()
    start = time.time()
    args.inputPath          = '/home/nas2/biod/zhencaiwei/MultiChat-main/Datasets/ISSAAC_GT/' # replace with your own path
    args.use_cuda           = args.use_cuda and torch.cuda.is_available()

    args.outPath            = args.inputPath + 'Bg_CCC/'	
    args.spatialLocation    = args.inputPath + 'inputs/' + 'Coord.csv'
    args.annoFile           = args.inputPath + 'inputs/' + 'celltype_info.csv'
    args.pos_pair           = args.outPath + 'Spot_positive_pairs_shuffled.txt'
    args.Ligands_exp        = args.outPath + 'ligands_expression_shuffled.txt'
    args.Receptors_exp      = args.outPath + 'receptors_expression_shuffled.txt'
    
    args.patience = 15
    args.lr_cci = 0.001
    args.attn_drop = 0
    args.tau = 0.05
    
    args.lrp_strength_file = args.outPath + f'LRI_module_strength_run_{run_idx}.txt'
    
    # Run training
    MC.model_training.Train_CCC_model_parallel(args)
    
    duration = time.time() - start
    print(f'Finish training run {run_idx}, total time: {duration}s')

In [ ]:
'''Step2-2:Run MultiChat contrastive learning module to generate 10 sets of background data in parallel'''
if __name__ == '__main__':
    run_indices = list(range(1, 11))  # [1, 2, 3, ..., 10]
    
    with multiprocessing.Pool(processes=5) as pool:
        pool.map(run_training, run_indices)

### 2.3 Identify significant ligand–receptor pairs

>Now that we have both the L–R pair communication scores and the background distribution, we can perform statistical evaluations to filter out noise.
>
> **Single-cell level analysis**: We identify significant L–R pairs between individual communicating cells using Z-scores, with a default threshold of `1.6545`, corresponding to a one-sided p-value of `0.05` under the standard normal distribution.
>
> **Cell-type level analysis**: We aggregate the single-cell level results to identify robust and generalized communication patterns between different functional cell clusters. By default, the aggregation is performed using the mean, while alternative strategies (e.g., median or sum) can be specified via parameters.

**Identify single-cell level significant L–R pairs**

In [ ]:
base_path = '/home/nas2/biod/zhencaiwei/MultiChat-main/Datasets/ISSAAC_GT/' # replace with your own path
background_inter_df = MC.tl.load_background_inter(base_path+'Bg_CCC/', file_pattern="LRI_module_strength_run_*.txt")
background_inter_df.to_csv(base_path + 'Bg_CCC/LRI_module_strength_concat.txt', sep="\t", index=True)
sample_inter_df = pd.read_csv(base_path+'CCC/LRI_module_strength.txt', sep='\t', index_col=0)  
lr_lst = sample_inter_df.columns.tolist() 
sub_background_inter_df = background_inter_df.loc[:, lr_lst] 

We identify significant L–R pairs using a statistical test. The parameter `alpha` specifies the significance threshold (default = `0.05`) and can be adjusted to control the stringency of the results. A smaller `alpha` leads to more stringent filtering.

In [ ]:
sig_LR_pair = MC.tl.Identify_significant_lr_pairs(
    background_inter_df=sub_background_inter_df,
    sample_inter_df=sample_inter_df,
    output_path=base_path+'CCC/Significant_LRs.csv',
    z_critical=None,
    alpha=0.05
)
sig_LR_pair.head()

**Identify cell-type level significant L–R pairs** 

The significant L–R pairs are identified for each target cell type, representing interactions received by the specified cell type.

In [ ]:
background_inter_df = MC.tl.load_background_inter(base_path+'Bg_CCC/', file_pattern="LRI_module_strength_run_*.txt") 
sample_inter_df = pd.read_csv(base_path+'CCC/LRI_module_strength.txt', sep='\t', index_col=0)   
cell_clus = pd.read_csv(base_path+'/inputs/celltype_info.csv', header=0, index_col=0, sep="\t") 
sig_LR_pair = pd.read_csv(base_path+'CCC/Significant_LRs.csv', header=0, sep=",")
sig_LR_pair_celltype = MC.tl.Identify_significant_lr_pairs_celltype(sig_LR_pair, cell_clus, agg_method='mean')
merged = sample_inter_df.join(cell_clus)   
sample_inter_ct_df = merged.groupby('celltype').mean() 
sample_inter_ct_df.index.name = None 
vola_LR_pair_celltype, vola_LR_pair_celltype_vscore,vola_LR_pair_celltype_bin,_ = MC.tl.Identify_volatile_lr_pairs_celltype(sample_inter_ct_df, threshold=2.5, method='mad')
outs_path = base_path+'CCC/Significant_LRs_ct_concat.csv' 
sig_LR_pair_celltype_concat = MC.tl.Identify_concat_lr_pairs_celltype(sig_LR_pair_celltype,vola_LR_pair_celltype,outs_path)

The significant ligand–receptor pairs are identified for each source–target cell type pair, using the `Nei_adj.csv`

Therefore, we first identify significant ligand–receptor pairs with source and target annotation at the single-cell level.

In [ ]:
db = pd.read_csv(f'{base_path}inputs/LRpairDB_merge_filt1.csv', index_col=None, sep=',')
cell_clus.rename(columns={'celltype': 'cell_type'}, inplace=True)
MC.tl.sig_LR_with_source_target(base_path,db,cell_clus)

We then aggregate single-cell level significant L–R pairs to derive source–target cell type-level results.

In [ ]:
sig_LR_res = pd.read_csv(base_path + f'CCC/Significant_LRs_res.csv', index_col=None)
sig_LR_res_celltype = MC.tl.Identify_significant_lr_pairs_celltypes(sig_LR_res, agg_method='mean', min_cells_count=10)
sig_LR_res_celltype.to_csv(base_path + f'CCC/Significant_LRs_res_celltype.csv', index=False)
sig_LR_res_celltype.head()